# SMART Dataset — Exploratory Data Analysis v2

**175 ICU patients · Long-format time series · No labels yet**

Each patient file = one CSV with one row per measurement: `Feature | Value | Unit | Time`

All plots and CSVs are saved to `assets/`

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR   = Path('data')
ASSETS_DIR = Path('assets_2')
ASSETS_DIR.mkdir(exist_ok=True)  # creates assets/ folder if it does not exist

patient_files = sorted(DATA_DIR.glob('ID_*.csv'))

print(f'Patient files found : {len(patient_files)}')
print(f'Saving outputs to  : {ASSETS_DIR.resolve()}')

Patient files found : 175
Saving outputs to  : /Users/shwetabambal/Documents/myrepos/SAH-hiwi/assets_2


---
## 1 · Load all patient files

In [5]:
all_dfs     = []
load_errors = []

for f in patient_files:
    try:
        df = pd.read_csv(f, encoding='latin-1', parse_dates=['Time'])
        # encoding='latin-1' is required because German medical units (µg, µmol) use
        # Latin-1 characters that are invalid in the default UTF-8 encoding
        df['patient_id'] = f.stem   # adds column 'ID_001', 'ID_002', etc.
        all_dfs.append(df)
    except Exception as e:
        load_errors.append((f.stem, str(e)))

df_all = pd.concat(all_dfs, ignore_index=True)

print(f'Patients loaded successfully : {len(all_dfs)}')
print(f'Patients failed to load      : {len(load_errors)}')
if load_errors:
    for pid, err in load_errors:
        print(f'  ERROR {pid}: {err}')

print(f'\nTotal measurement rows : {len(df_all):,}')
print(f'Columns                : {list(df_all.columns)}')
print()
print('--- INSIGHT ---')
print('Each row is a single measurement for one patient at one timestamp.')
print('Different patients have very different numbers of rows depending on how long')
print('they were monitored and how many different things were measured.')

df_all.head(6)

Patients loaded successfully : 175
Patients failed to load      : 0

Total measurement rows : 1,945,892
Columns                : ['Feature', 'Value', 'Unit', 'Time', 'patient_id']

--- INSIGHT ---
Each row is a single measurement for one patient at one timestamp.
Different patients have very different numbers of rows depending on how long
they were monitored and how many different things were measured.


,Feature,Value,Unit,Time,patient_id
0,CO2 Konzentration Endtidal in mmHg,7.0,mmHg,2021-03-21 06:08:00,ID_002
1,CO2 Konzentration Endtidal in mmHg,48.0,mmHg,2021-03-21 06:23:00,ID_002
2,CO2 Konzentration Endtidal in mmHg,48.0,mmHg,2021-03-21 06:38:00,ID_002
3,CO2 Konzentration Endtidal in mmHg,59.0,mmHg,2021-03-21 06:53:00,ID_002
4,CO2 Konzentration Endtidal in mmHg,58.0,mmHg,2021-03-21 07:08:00,ID_002
5,CO2 Konzentration Endtidal in mmHg,54.0,mmHg,2021-03-21 11:49:00,ID_002


---
## 2 · Per-patient summary table

**What is `n_rows`?**  
The total number of individual measurements recorded for that patient.  
A patient with only 30 rows means barely any data was recorded — too little to train a model on.

**What does `duration_days < 1` mean?**  
All measurements for that patient happened within the same day (< 24 hours).  
This could mean the patient died or was transferred very quickly, or it is a data export error.  
Either way, a single day of data gives us almost no time-series information.

**Why flag these thresholds?**  
Before building any model, we need to decide: *which patients have enough data to be useful?*  
Patients with very few rows or very short duration will add noise rather than signal.

In [ ]:
ps = df_all.groupby('patient_id').agg(
    n_rows            = ('Feature', 'count'),       # total measurements
    n_unique_features = ('Feature', 'nunique'),      # how many distinct things were measured
    first_time        = ('Time', 'min'),             # earliest timestamp
    last_time         = ('Time', 'max'),             # latest timestamp
).reset_index()

# Duration = time from first to last measurement
ps['duration_days'] = (ps['last_time'] - ps['first_time']).dt.total_seconds() / 86400
ps['admission_year'] = ps['first_time'].dt.year

# ICU stays beyond 180 days are medically implausible → flag as export errors
ps['export_error'] = ps['duration_days'] > 180

print('=== Dataset-wide statistics ===')
print(ps[['n_rows', 'duration_days', 'n_unique_features']].describe().round(1))

# --- Export errors ---
export_err = ps[ps['export_error']]
print(f'\n[!] Suspected EXPORT ERRORS (duration > 180 days): {len(export_err)} patients')
print(export_err[['patient_id', 'duration_days', 'n_rows']].to_string(index=False))

# --- Sparse patients: fewer than 100 rows ---
sparse_rows = ps[ps['n_rows'] < 100].sort_values('n_rows')
print(f'\n[!] Patients with FEWER THAN 100 rows (very little data): {len(sparse_rows)}')
print('    These patients have so few measurements that including them in a model')
print('    would likely add noise. Consider excluding them.')
print(sparse_rows[['patient_id', 'n_rows', 'duration_days', 'n_unique_features']].to_string(index=False))

# --- Short duration: less than 1 day ---
short_dur = ps[ps['duration_days'] < 1].sort_values('duration_days')
print(f'\n[!] Patients with LESS THAN 1 DAY of data: {len(short_dur)}')
print('    All their measurements happened within the same 24-hour window.')
print('    There is almost no temporal signal available for these patients.')
print(short_dur[['patient_id', 'duration_days', 'n_rows', 'n_unique_features']].to_string(index=False))

print()
print('--- INSIGHT ---')
print(f'Out of {len(ps)} patients:')
print(f'  · {len(export_err)} have suspiciously long durations (likely export bugs)')
print(f'  · {len(sparse_rows)} have fewer than 100 measurements (too sparse)')
print(f'  · {len(short_dur)} have all data within a single day (no time-series value)')
print(f'  · Median monitoring duration is {ps["duration_days"].median():.1f} days')
print(f'  · Median measurements per patient is {ps["n_rows"].median():.0f} rows')

---
## 3 · Patient-level plots

> **Note on histograms:** The y-axis shows *how many patients* fall in each bar (bin).  
> For example, if a bar at x=5000 rows has height 30, it means 30 patients each have ~5000 rows.  
> All 175 patients are represented — they are just spread across many bars.

In [ ]:
# PLOT 1: How many measurement rows does each patient have?
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(ps['n_rows'], bins=40, color='steelblue', edgecolor='white')
ax.axvline(ps['n_rows'].median(), color='red', linestyle='--',
           label=f'Median = {ps["n_rows"].median():.0f} rows')
ax.axvline(100, color='orange', linestyle=':', label='100-row threshold')
ax.set_title('Rows per Patient  (y = number of patients in that range)')
ax.set_xlabel('Number of measurement rows')
ax.set_ylabel('Number of patients')
ax.legend()
plt.tight_layout()
plt.savefig(ASSETS_DIR / 'fig_01_rows_per_patient.png', dpi=120)
plt.show()

print('Saved → assets/fig_01_rows_per_patient.png')
print()
print('--- INSIGHT ---')
print(f'Most patients have between {ps["n_rows"].quantile(0.25):.0f} and {ps["n_rows"].quantile(0.75):.0f} rows (middle 50%).')
print(f'A few patients have very many rows (long ICU stays with frequent monitoring).')
print(f'Patients below the orange line (<100 rows) have almost no data.')

In [ ]:
# PLOT 2: How many days was each patient monitored?
# We exclude export-error patients from the histogram so the scale is readable.
clean   = ps[~ps['export_error']]['duration_days']
bad_ps  = ps[ps['export_error']]

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(clean, bins=40, color='coral', edgecolor='white')
ax.axvline(clean.median(), color='red', linestyle='--',
           label=f'Median = {clean.median():.1f} days')
ax.axvline(1, color='orange', linestyle=':', label='1-day cutoff')
ax.set_title(f'Monitoring Duration per Patient  (excl. {len(bad_ps)} export-error patients)')
ax.set_xlabel('Days from first to last measurement')
ax.set_ylabel('Number of patients')
ax.legend()
excluded_note = '  |  '.join(f"{r.patient_id}: {r.duration_days:.0f} d" for _, r in bad_ps.iterrows())
fig.text(0.5, -0.04, f'Export-error patients excluded from chart: {excluded_note}',
         ha='center', fontsize=7, color='gray')
plt.tight_layout()
plt.savefig(ASSETS_DIR / 'fig_02_monitoring_duration.png', dpi=120, bbox_inches='tight')
plt.show()

print('Saved → assets/fig_02_monitoring_duration.png')
print()
print('--- INSIGHT ---')
print(f'Median ICU monitoring duration is {clean.median():.1f} days — a realistic ICU stay.')
print(f'{(clean < 1).sum()} patients have all data within 1 day (orange line) — minimal time-series value.')
print(f'Export-error patients excluded: {list(bad_ps["patient_id"])} (durations > 180 days are unrealistic).')

In [ ]:
# PLOT 3: How many distinct features (vitals/labs) were measured per patient?
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(ps['n_unique_features'], bins=30, color='seagreen', edgecolor='white')
ax.axvline(ps['n_unique_features'].median(), color='red', linestyle='--',
           label=f'Median = {ps["n_unique_features"].median():.0f} features')
ax.set_title('Unique Features Measured per Patient')
ax.set_xlabel('Number of distinct features')
ax.set_ylabel('Number of patients')
ax.legend()
plt.tight_layout()
plt.savefig(ASSETS_DIR / 'fig_03_unique_features_per_patient.png', dpi=120)
plt.show()

print('Saved → assets/fig_03_unique_features_per_patient.png')
print()
print('--- INSIGHT ---')
print(f'Median patient has {ps["n_unique_features"].median():.0f} distinct features measured.')
print(f'Min: {ps["n_unique_features"].min()}, Max: {ps["n_unique_features"].max()}')
print('Patients with very few unique features may lack key clinical variables needed for modelling.')

In [ ]:
# PLOT 4: In which year was each patient admitted?
# Different years = patients admitted at different real-world times (normal for a multi-year study).
fig, ax = plt.subplots(figsize=(8, 4))
ps['admission_year'].value_counts().sort_index().plot(
    kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Patients by Admission Year')
ax.set_xlabel('Year of first recorded measurement')
ax.set_ylabel('Number of patients')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(ASSETS_DIR / 'fig_04_admission_years.png', dpi=120)
plt.show()

print('Saved → assets/fig_04_admission_years.png')
print()
print('--- INSIGHT ---')
year_counts = ps['admission_year'].value_counts().sort_index()
for yr, cnt in year_counts.items():
    print(f'  {yr}: {cnt} patients')
print('Different years are expected — this is a multi-year retrospective clinical study.')
print('For modelling, timestamps should be converted to hours-since-admission (relative time),')
print('not used as absolute calendar dates.')

In [ ]:
# PLOT 5: Duration scatter — every patient as a dot, sorted shortest to longest.
# Export-error patients are marked with a red X so we can see their IDs.
sorted_ps = ps.sort_values('duration_days').reset_index(drop=True)
normal    = sorted_ps[~sorted_ps['export_error']]
bad       = sorted_ps[sorted_ps['export_error']]

fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(normal.index, normal['duration_days'],
           s=20, alpha=0.6, color='coral', label='Valid patients')
ax.scatter(bad.index, bad['duration_days'],
           s=80, color='red', marker='x', zorder=5,
           label=f'Suspected export errors  (n={len(bad)})')
for _, row in bad.iterrows():
    ax.annotate(row['patient_id'],
                xy=(row.name, row['duration_days']),
                xytext=(6, 4), textcoords='offset points',
                fontsize=7, color='red')
ax.axhline(1,   color='orange', linestyle=':', linewidth=1, label='1-day cutoff')
ax.axhline(180, color='purple', linestyle=':', linewidth=1, label='180-day threshold')
ax.set_title('Duration per Patient — Each Dot = One Patient (Sorted Ascending)')
ax.set_xlabel('Patient rank (1 = shortest stay)')
ax.set_ylabel('Days in dataset')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(ASSETS_DIR / 'fig_05_duration_scatter.png', dpi=120)
plt.show()

print('Saved → assets/fig_05_duration_scatter.png')
print()
print('--- INSIGHT ---')
print(f'Most patients (coral dots) cluster below 60 days — a plausible ICU stay.')
print(f'Red X patients have durations that are medically impossible:')
for _, row in bad.iterrows():
    print(f'  {row["patient_id"]}: {row["duration_days"]:.0f} days')
print('These should be reported to the supervisor as data export issues.')

---
## 4 · Feature coverage — how many patients have each feature?

In [ ]:
feat_cov = pd.read_csv(DATA_DIR / 'Feature_File_Count_Summary.csv', encoding='latin-1')
feat_cov.columns = feat_cov.columns.str.strip()
feat_cov = feat_cov.sort_values('File_Count', ascending=False).reset_index(drop=True)
feat_cov['coverage_pct'] = feat_cov['File_Count'] / len(patient_files) * 100

high   = feat_cov[feat_cov['coverage_pct'] > 90]
medium = feat_cov[(feat_cov['coverage_pct'] >= 50) & (feat_cov['coverage_pct'] <= 90)]
low    = feat_cov[feat_cov['coverage_pct'] < 50]

print(f'Total unique features          : {len(feat_cov)}')
print(f'High coverage  (>90% patients) : {len(high)}   ← safe to use in models')
print(f'Medium coverage (50–90%)       : {len(medium)}  ← usable with imputation')
print(f'Low coverage   (<50% patients) : {len(low)}   ← likely too sparse')
print()
print('Top 20 features by coverage:')
print(feat_cov[['Feature','File_Count','coverage_pct']].head(20).to_string(index=False))
print()
print('--- INSIGHT ---')
print(f'Only {len(high)} features appear in >90% of patients.')
print('These are the most reliable features for modelling.')
print(f'{len(low)} features appear in fewer than half of all patients — they will create')
print('many missing values if included and should likely be excluded or handled carefully.')

In [ ]:
# PLOT 6: Distribution of how well-covered each feature is
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(feat_cov['coverage_pct'], bins=20, color='mediumpurple', edgecolor='white')
ax.axvline(90, color='red',    linestyle='--', label=f'>90%: {len(high)} features')
ax.axvline(50, color='orange', linestyle='--', label=f'<50%: {len(low)} features')
ax.set_title('Feature Coverage — What % of Patients Have Each Feature?')
ax.set_xlabel('% of patients that have this feature')
ax.set_ylabel('Number of features')
ax.legend()
plt.tight_layout()
plt.savefig(ASSETS_DIR / 'fig_06_feature_coverage_histogram.png', dpi=120)
plt.show()

print('Saved → assets/fig_06_feature_coverage_histogram.png')
print()
print('--- INSIGHT ---')
print('Two peaks visible: many features appear in nearly ALL patients (right side)')
print('and many features appear in only a FEW patients (left side).')
print('This bimodal pattern is common in clinical data — core measurements vs. optional tests.')

In [ ]:
# PLOT 7: Which are the 30 most commonly measured features?
top30 = feat_cov.head(30)

fig, ax = plt.subplots(figsize=(10, 9))
bars = ax.barh(top30['Feature'][::-1], top30['coverage_pct'][::-1], color='steelblue')
ax.axvline(90, color='red',    linestyle='--', alpha=0.8, label='90% threshold')
ax.axvline(50, color='orange', linestyle='--', alpha=0.8, label='50% threshold')
ax.set_xlabel('% of patients with this feature')
ax.set_title('Top 30 Most Commonly Measured Features')
ax.legend()
plt.tight_layout()
plt.savefig(ASSETS_DIR / 'fig_07_top30_features.png', dpi=120)
plt.show()

print('Saved → assets/fig_07_top30_features.png')
print()
print('--- INSIGHT ---')
above90 = top30[top30['coverage_pct'] > 90]['Feature'].tolist()
print(f'Features above 90% coverage (core feature candidates):')
for f in above90:
    print(f'  · {f}')

---
## 5 · Value quality and distributions

We convert all `Value` entries to numbers. Entries that cannot be converted (e.g. text like `'n.a.'` or `'>'`) become `NaN`.  
Each top feature then gets its own plot.

In [ ]:
df_all['Value_num'] = pd.to_numeric(df_all['Value'], errors='coerce')

non_num_pct = df_all['Value_num'].isna().mean() * 100
non_num_examples = df_all[df_all['Value_num'].isna()]['Value'].value_counts().head(15)

print(f'Non-numeric value rate: {non_num_pct:.2f}%')
print(f'({df_all["Value_num"].isna().sum():,} out of {len(df_all):,} rows cannot be parsed as numbers)')
print()
print('Most common non-numeric values:')
print(non_num_examples.to_string())
print()
print('--- INSIGHT ---')
print(f'{non_num_pct:.1f}% of values are non-numeric.')
print('Some of these are legitimate categories (e.g. blood type), some are measurement symbols')
print('(e.g. ">100" means "above detection limit"), and some may be data entry errors.')
print('These need to be handled before modelling (e.g. encode, cap, or drop).')

In [ ]:
# Compute per-feature statistics (only numeric values)
feature_stats = df_all.groupby('Feature')['Value_num'].agg(
    count  = 'count',
    mean   = 'mean',
    std    = 'std',
    min    = 'min',
    p25    = lambda x: x.quantile(0.25),
    median = 'median',
    p75    = lambda x: x.quantile(0.75),
    max    = 'max',
    n_null = lambda x: x.isna().sum(),
).reset_index()

feature_stats['null_pct'] = (feature_stats['n_null'] / len(df_all) * 100).round(2)
feature_stats = feature_stats.sort_values('count', ascending=False).reset_index(drop=True)

print('Top 20 features by total measurement count:')
print(feature_stats[['Feature','count','mean','std','min','median','max']].head(20).to_string(index=False))
print()
print('--- INSIGHT ---')
print('Features with high std relative to mean may have extreme outliers.')
print('Features where min << median or max >> median are worth investigating.')

In [ ]:
# PLOTS 8–19: One histogram per top-12 feature, each saved as a separate image
top12 = feature_stats.head(12)['Feature'].tolist()

for i, feat in enumerate(top12, start=8):
    vals      = df_all[df_all['Feature'] == feat]['Value_num'].dropna()
    unit_mode = df_all[df_all['Feature'] == feat]['Unit'].dropna().mode()
    unit_str  = unit_mode.iloc[0] if len(unit_mode) else ''
    skewness  = vals.skew()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(vals, bins=40, color='steelblue', edgecolor='white')
    ax.axvline(vals.mean(),   color='orange', linestyle='--', linewidth=1.2,
               label=f'Mean = {vals.mean():.2f}')
    ax.axvline(vals.median(), color='red',    linestyle='-',  linewidth=1.2,
               label=f'Median = {vals.median():.2f}')
    ax.set_title(f'{feat}  [{unit_str}]  ·  n={len(vals):,}')
    ax.set_xlabel(f'Value  ({unit_str})')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    plt.tight_layout()

    safe  = feat.replace('/', '_').replace(' ', '_').replace('.','').replace(',','')[:48]
    fname = f'fig_{i:02d}_dist_{safe}.png'
    plt.savefig(ASSETS_DIR / fname, dpi=120)
    plt.close()

    skew_note = 'right-skewed (long tail of high values)' if skewness > 1 else \
                'left-skewed (long tail of low values)'   if skewness < -1 else \
                'roughly symmetric'
    print(f'[fig_{i:02d}] {feat}')
    print(f'         Unit: {unit_str}  |  n={len(vals):,}  |  median={vals.median():.2f}')
    print(f'         Range: {vals.min():.2f} – {vals.max():.2f}  |  Distribution: {skew_note}')
    print()

print(f'Saved {len(top12)} individual distribution plots to assets/')

---
## 6 · Outlier detection (IQR × 3 rule)

A value is flagged as an outlier if it falls more than 3× the interquartile range (IQR) below Q1 or above Q3.  
This is a conservative threshold — we only flag extreme values, not mild ones.

In [ ]:
def iqr_outlier_stats(s, factor=3.0):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr    = q3 - q1
    lo, hi = q1 - factor * iqr, q3 + factor * iqr
    n_out  = int(((s < lo) | (s > hi)).sum())
    return n_out, round(n_out / len(s) * 100, 2), round(lo, 3), round(hi, 3)

rows = []
for feat in feature_stats['Feature']:
    vals = df_all[df_all['Feature'] == feat]['Value_num'].dropna()
    if len(vals) < 10:
        continue
    n_out, pct, lo, hi = iqr_outlier_stats(vals)
    rows.append(dict(
        Feature=feat, n_values=len(vals), n_outliers=n_out, outlier_pct=pct,
        iqr_lower=lo, iqr_upper=hi,
        actual_min=round(vals.min(), 3), actual_max=round(vals.max(), 3)
    ))

outlier_df = pd.DataFrame(rows).sort_values('outlier_pct', ascending=False).reset_index(drop=True)

print('Features with highest extreme-outlier rates (IQR × 3):')
print(outlier_df.head(20).to_string(index=False))
print()
print('--- INSIGHT ---')
high_outlier = outlier_df[outlier_df['outlier_pct'] > 5]
print(f'{len(high_outlier)} features have >5% of values flagged as extreme outliers.')
print('These may indicate measurement errors, unit inconsistencies, or genuinely extreme clinical events.')
print('They should be reviewed with the clinical team before modelling.')

In [ ]:
# PLOT 20: Which features have the most outliers?
top_out = outlier_df.head(20)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_out['Feature'][::-1], top_out['outlier_pct'][::-1], color='tomato')
ax.axvline(5, color='red', linestyle='--', alpha=0.7, label='5% concern threshold')
ax.set_xlabel('% of values flagged as extreme outliers (IQR × 3)')
ax.set_title('Top 20 Features by Outlier Rate')
ax.legend()
plt.tight_layout()
plt.savefig(ASSETS_DIR / 'fig_20_outlier_rates.png', dpi=120)
plt.show()

print('Saved → assets/fig_20_outlier_rates.png')
print()
print('--- INSIGHT ---')
print('Features on the right side of the chart need careful handling.')
print('Options: clip values at IQR bounds, log-transform, or discuss with clinicians')
print('whether extreme values represent true physiology or recording errors.')

---
## 7 · Feature presence heatmap

Each row = one feature. Each column = one patient.  
Blue = feature was measured for that patient. White = not measured.

In [ ]:
# PLOT 21: Which patients have which features?
top40 = feat_cov.head(40)['Feature'].tolist()

presence = (
    df_all[df_all['Feature'].isin(top40)]
    .groupby(['patient_id', 'Feature'])
    .size()
    .unstack(fill_value=0)
    .gt(0)
    .astype(int)
)

fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(presence.T, cmap='Blues', linewidths=0, ax=ax,
            cbar_kws={'label': 'Feature measured (1=yes, 0=no)'},
            xticklabels=False)
ax.set_title('Feature Presence Matrix — Top 40 Features × 175 Patients')
ax.set_ylabel('Feature')
ax.set_xlabel('Each column = one patient')
plt.tight_layout()
plt.savefig(ASSETS_DIR / 'fig_21_feature_presence_heatmap.png', dpi=120)
plt.show()

print('Saved → assets/fig_21_feature_presence_heatmap.png')
print()
print('--- INSIGHT ---')
coverage_per_feature = presence.mean() * 100
full_rows  = (presence.sum(axis=1) == len(top40)).sum()
print(f'Features with 100% presence across all patients: {(coverage_per_feature == 100).sum()}')
print(f'Patients with ALL top-40 features present: {full_rows}')
print('White gaps in the heatmap = missing features for that patient.')
print('A mostly-blue heatmap = good data completeness.')

---
## 8 · Save all summary CSVs

In [ ]:
ps.to_csv(ASSETS_DIR / 'summary_patients.csv',          index=False)
feature_stats.to_csv(ASSETS_DIR / 'summary_features.csv',   index=False)
outlier_df.to_csv(ASSETS_DIR / 'summary_outliers.csv',      index=False)
feat_cov.to_csv(ASSETS_DIR / 'summary_feature_coverage.csv', index=False)

print('All outputs saved to assets/:')
print()
all_outputs = sorted(ASSETS_DIR.iterdir())
pngs = [f for f in all_outputs if f.suffix == '.png']
csvs = [f for f in all_outputs if f.suffix == '.csv']

print(f'  Plots ({len(pngs)}):')
for f in pngs:
    print(f'    {f.name}')

print(f'\n  CSVs ({len(csvs)}):')
for f in csvs:
    print(f'    {f.name}  ({f.stat().st_size/1024:.1f} KB)')

---
## 9 · Key findings — fill in after running

| # | Question | Finding |
|---|---|---|
| 1 | Total patients loaded | 175 |
| 2 | Total measurement rows | — |
| 3 | Median rows per patient | — |
| 4 | Patients with <100 rows (IDs) | — |
| 5 | Patients with <1 day data (IDs) | — |
| 6 | Suspected export errors (IDs) | — |
| 7 | Median monitoring duration | — |
| 8 | Total unique features | — |
| 9 | Features in >90% of patients | — |
| 10 | Features in <50% of patients | — |
| 11 | Non-numeric value rate | — |
| 12 | Top outlier feature | — |

### Questions to discuss with supervisor on Monday
1. Should we exclude patients with <100 rows or <1 day of data?
2. Are the export-error patients (ID_146, ID_143, ID_133, ID_038) real or bugs?
3. What is the minimum feature coverage % we accept for the core feature set?
4. How should non-numeric values be handled (encode, cap, drop)?
5. Should outlier values be clipped or kept for clinical review?